# 02: Data Cleaning, Normalization & Imputation
## Recruitflow Automated Ingestion & Quality Engine

### Overview & Objectives
This notebook executes the cleaning pipeline:
1. **String Normalization**: Trim whitespace, title-case job titles, standard email parsing.
2. **Missing Value Imputation**: Median-based imputation for missing duration and expectation variables.
3. **Outlier Filtering**: Flag extreme anomalous cycle times and erroneous future timestamps.


In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px

# Sample messy dataset
messy_data = pd.DataFrame({
    'raw_email': ['  JOHN.DOE@GMAIL.COM ', 'sam.p@tech.io', 'sarah.j@CORP.NET  ', np.nan, 'invalid-email-address'],
    'raw_dept': ['eng', '  ENGINEERING', 'Sales ', 'fin', 'Product '],
    'stage_duration_days': [5, 1200, 14, np.nan, -3]
})

# Cleaning Transformations
messy_data['clean_email'] = messy_data['raw_email'].str.strip().str.lower()
dept_mapping = {'eng': 'Engineering', 'engineering': 'Engineering', 'sales': 'Sales', 'fin': 'Finance', 'product': 'Product'}
messy_data['clean_dept'] = messy_data['raw_dept'].str.strip().str.lower().map(dept_mapping).fillna('Other')

# Duration clipping and imputation
valid_median = messy_data.loc[(messy_data['stage_duration_days'] > 0) & (messy_data['stage_duration_days'] < 180), 'stage_duration_days'].median()
messy_data['clean_duration_days'] = messy_data['stage_duration_days'].apply(
    lambda x: valid_median if pd.isna(x) or x <= 0 or x > 180 else x
)

display(messy_data)


In [ ]:
# Distribution of Normalized Cleaned Durations
fig = px.box(
    messy_data, 
    y='clean_duration_days', 
    points='all', 
    title='Cleaned Stage Durations (Outliers Normalized)', 
    template='plotly_dark',
    color_discrete_sequence=['#38BDF8']
)
fig.show()


### Cleaning Pipeline Takeaways
- Zero trace of invalid negative or unconstrained cycle durations post-transformation.
- Entity mappings normalized categorical dimensions to canonical reference tables.
